In [ ]:
import os; os.environ["CUDA_LAUNCH_BLOCKING"]="1"
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels
kill_other_ipykernels(force=True)
import trajectory_data
import matplotlib.pyplot as plt
from nocode_robot_programming.state_decision.dataloader import TrajectoryDataset
from nocode_robot_programming.state_decision.dino_model import DINOStateDecider
# from nocode_robot_programming.state_decision.dino_model_v2 import DINOFeaturePresence
from nocode_robot_programming.state_decision.dino_model_v3 import DINOFeaturePresence
from nocode_robot_programming.state_decision.SIFT_model import StateDeciderSIFT
from nocode_robot_programming.state_decision.AEGP_model import AEGP
from nocode_robot_programming.state_decision.state_decider import StateDeciderBase
from nocode_robot_programming.state_decision.utils import Filename
from gesture_detector.utils import pretty_confusion_matrix
import torch
from torch.utils.data import DataLoader
import numpy as np

from trajectory_data.skill_visualizer import play_video
from nocode_robot_programming.state_decision.utils import add_tag
from nocode_robot_programming.state_decision.dataloader import ImageDatasetView, saved_img_processing
from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt, show_gray_video_cuda

from copy import deepcopy
seed = 50
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

from nocode_robot_programming.state_decision.dataset_d1 import d1_peg_pick, dupl
datafileloader = TrajectoryDataset(trajectory_data.package_path)
datasets = d1_peg_pick(datafileloader)

# 1. Evaluation of AE-GP method on Dataset `d1`

- function `get_dataset_view` loads the recordings

In [ ]:
display(show_gray_video_cuda(datasets[0][0].X, fps=20, scale=5))
# d_train.play_video(fps=10)
# d_test.play_video(fps=10)

In [ ]:
# for task_name in dataset.tasks.keys():
for d_train, d_test, d_text in datasets:
    print(d_text)
    for decider in [
            # DINOFeaturePresence(percentile_keep=0.0),
            # DINOStateDecider(dino_variant = "dinov2_vitl14", percent_keep=0),
            # StateDeciderSIFT(),
            AEGP(),
        ]:
        decider.train(d_train.X, d_train.y_int, d_train.y_cls)

        y_pred = decider.predict_many(d_train.X); ipt.save()
        pretty_confusion_matrix.pp_matrix_from_string_data(d_train.y_names, y_pred, name=f"d_train,{decider}", show=False, figsize=None); ipt.save()

        y_pred = decider.predict_many(d_test.X); ipt.save()
        pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False, figsize=None); ipt.save()

        ipt.show()

        rr = range(0,d_train.X.size(0),5) # reconstructed range - image selections
        reconstructed_images = decider.videoembedder.model.forward(d_train.X[rr].unsqueeze(1)).squeeze()
        display(show_gray_video_cuda(torch.concatenate([d_train.X[rr], reconstructed_images], dim=2), fps=20, scale=5))

In [ ]:
# for task_name in dataset.tasks.keys():
for d_train, d_test, d_text in datasets:
    print("--------------------------------------------")
    print(d_text)
    for decider in [
            # DINOFeaturePresence(percentile_keep=0.0),
            # DINOStateDecider(dino_variant = "dinov2_vitl14", percent_keep=0),
            # StateDeciderSIFT(),
            AEGP(),
        ]:
        for i in [10,20]:
            print(f"  {i}x  ")
            d_train_ = dupl(d_train, n=i)
            decider.train(d_train_.X, d_train_.y_int, d_train_.y_cls)

            y_pred = decider.predict_many(d_train_.X); ipt.save()
            pretty_confusion_matrix.pp_matrix_from_string_data(d_train_.y_names, y_pred, name=f"d_train,{decider}", show=False, figsize=None); ipt.save()

            y_pred = decider.predict_many(d_test.X); ipt.save()
            pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False, figsize=None); ipt.save()

            ipt.show()

            rr = range(0,d_train_.X.size(0),5) # reconstructed range - image selections
            reconstructed_images = decider.videoembedder.model.forward(d_train_.X[rr].unsqueeze(1)).squeeze()
            display(show_gray_video_cuda(torch.concatenate([d_train_.X[rr], reconstructed_images], dim=2), fps=20, scale=5))

In [ ]:
y_pred = decider.predict_many(d_test.X); ipt.save()
pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False, figsize=None); ipt.save()
ipt.show()

# Just utils

In [ ]:
task_name = "d1_peg_pick"
names_and_tags = [f"{name}: {tag}" for name,tag in zip(datafileloader.tasks[task_name]['names'], datafileloader.tasks[task_name]['tags'])]
names_and_tags

In [ ]:
[n for n in datafileloader.tasks['d1_peg_pick']['names'] if "trial" not in n or n == 'd1_peg_pick_trial_3']

In [ ]:
[n for n in datafileloader.tasks['d1_peg_pick']['names'] if "trial" in n and n != 'd1_peg_pick_trial_3']